<a href="https://colab.research.google.com/github/vifirsanova/tutorials/blob/main/fine-tuning/clip-tuning-ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Туториал по обучению CLIP на основе материала: https://github.com/mlfoundations/open_clip/discussions/911



## Загрузка библиотек

In [ ]:
# !pip install av torchcodec open_clip_torch tqdm datasets

import open_clip
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

## Загрузка датасета

In [ ]:
dataset = load_dataset("ваш датасет")
full_dataset = dataset['train']  # Обучающая выборка

## Загрузка pre-trained модели

In [ ]:
model_name = 'EVA02-B-16' # выбранная модель
pretrained = 'merged2b_s8b_b131k' # можно тут убрать или подставить ссылку на предобученные веса из HF
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
tokenizer = open_clip.get_tokenizer(model_name)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Base model {model_name} loaded on {device}")

# Текстовые метки
text_labels = [str(i) for i in range(8)]
text = tokenizer(text_labels).to(device)

## Преодобработка датасета

In [ ]:
# Делим дату на train/validation выборки
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
print(f"Train samples: {train_size}, Validation samples: {val_size}")

### Немножко ООП: нужно создать класс для кастомного датасета

In [ ]:
class GestureVideoDataset(Dataset):
    def __init__(self, subset_data, transform, is_train=True):
        self.data = subset_data
        self.transform = transform
        self.is_train = is_train  # Это мы оставляем на случай, если вы захотите расширить выборку

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        video_data = sample['video']
        label = sample['label']

        # Извлекаем центральный фрейм из видео
        # (это наиболее простой подход, чтобы не анализировать каждый кадр, хотя он ограничивает нас)
        frames = []
        try:
            for i, frame_tensor in enumerate(video_data):
                # Размер видео - это тензоры размера (C, H, W)
                arr = frame_tensor.permute(1, 2, 0).cpu().numpy()
                # Конвертируем в uint8 ([0,1])
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                else:
                    arr = arr.astype(np.uint8)
                frame = Image.fromarray(arr)
                frames.append(frame)
        except Exception as e:
            print(f"Error extracting frames at index {idx}: {e}")
            # При ошибке можно вернуть сэмпл-заглушку, чтобы не ломать DataLoader
            # Для простоты мы просто вызовем сообщение об ошибке
            raise e

        if len(frames) == 0:
            raise ValueError(f"No frames extracted for sample {idx}")

        # Выбираем центральный фрейм
        middle_frame = frames[len(frames) // 2]

        # Препроцессинг модели CLIP
        image_tensor = self.transform(middle_frame)

        return image_tensor, label

### Дата-лоадер

Он делит ваши данные на небольшие пакеты, которые GPU сможет обработать

In [ ]:
train_loader = DataLoader(
    GestureVideoDataset(train_dataset, preprocess),
    batch_size=8,  # Делаем меньше, если GPU падает
    shuffle=True,
    num_workers=2  # Можно увеличить для скорости
)
val_loader = DataLoader(
    GestureVideoDataset(val_dataset, preprocess),
    batch_size=8,
    shuffle=False,
    num_workers=2
)

## Модель

Идея в том, что основные веса CLIP заморожены, мы обучаем только классификатор

In [ ]:
class CLIPFineTuner(nn.Module):
    def __init__(self, clip_model, num_classes):
        super().__init__()
        self.clip_model = clip_model
        # Замораживает CLIP vision encoder
        for param in self.clip_model.visual.parameters():
            param.requires_grad = False
        # Обучаем классификатор поверх энкодера
        self.classifier = nn.Linear(clip_model.visual.output_dim, num_classes)

    def forward(self, images):
        with torch.no_grad():
            # Достаём фичи из CLIP-энкодера
            features = self.clip_model.encode_image(images)
            # Фичи float32
            features = features.float()
        return self.classifier(features)

num_classes = 8
model_ft = CLIPFineTuner(model, num_classes).to(device)

## Функция потерь и оптимизатор

In [ ]:
criterion = nn.CrossEntropyLoss()
# Обучаем только параметры классификатора поверх CLIP
optimizer = optim.Adam(model_ft.classifier.parameters(), lr=1e-4)

## Обучение

In [ ]:
num_epochs = 5
print("\nStarting Fine-Tuning...")

for epoch in range(num_epochs):
    # Старт обучения
    model_ft.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_ft(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'Loss': f'{running_loss / (pbar.n + 1):.4f}'})

    avg_train_loss = running_loss / len(train_loader)

    # Оценка на валидации
    model_ft.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_ft(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100.0 * correct / total
    print(f'Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Acc = {val_acc:.2f}%')

## Сохраняем модель

In [ ]:
torch.save(model_ft.state_dict(), 'clip_finetuned_gesture.pth')

## Проверка производительности на валидации

Здесь нужно будет заменить валидацию на тестовую выборку, а затем добавить подсчет метрик оценки Precision, Recall, F-Score

In [ ]:
model_ft.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validating"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_ft(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_acc = 100.0 * correct / total
print(f'Final Validation Accuracy: {final_acc:.2f}%')